# Kabyle ASR with Whisper

### A - Installs

In [ ]:
!pip install --upgrade pip
!pip install datasets[audio] transformers accelerate evaluate jiwer soundfile
!pip install torchcodec

### B - Imports

In [ ]:
import evaluate
import os
import subprocess
import torch
import torchcodec

from datasets     import load_dataset, DatasetDict, Audio
from dataclasses  import dataclass
from google.colab import drive
from google.colab import runtime
from typing       import Any, Dict, List, Union
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, pipeline

## Part One: Training

### 1 - Loading and Inspecting the Dataset
The dataset “TutlaytAI/kabyle_asr” is from Hugging Face.

In [ ]:
dataset = load_dataset("TutlaytAI/kabyle_asr")
print(dataset)

### 2 - Preprocessing the Audio Data
Resampling to 16 kHz the audio files.

In [ ]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

### 3 - Loading the Whisper Processor (Feature Extractor + Tokenizer)

In [ ]:
model_name = "openai/whisper-small"
processor  = WhisperProcessor.from_pretrained(model_name, task="transcribe", language="kabyle")

### 4 - Features Extraction and Labels Encoding

In [ ]:
def prepare_batch(batch):
    """
    Preprocesses a batch for ASR training by extracting input features from audio and encoding transcription labels.

    Args:
        batch (dict): A dictionary containing the keys "audio" (with "array" and "sampling_rate") and "Text".

    Returns:
        dict: The updated batch with added "input_features" (model-ready audio features) and "labels" (tokenized transcription).
    """
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    batch["labels"] = processor.tokenizer(batch["Text"]).input_ids

    return batch

# Apply for every batch in the dataset and remove unnecessary columns so only audio, text, input_features, labels remain
dataset = dataset.map(
    prepare_batch,
    remove_columns=[c for c in dataset["train"].column_names if c not in ("audio", "Text")],
    num_proc=4,
)

### 5 - Defining the Data Collator
Pads and batches the input features and labels, masks padding tokens, and removes the beginning-of-sequence token if present.

In [ ]:
class DataCollator:
    def __init__(self, processor, decoder_start_token_id):
        """
        Initializes the data collator.

        Args:
            processor: The processor object containing the feature extractor and tokenizer.
            decoder_start_token_id (int): The token ID used to indicate the start of decoding.
        """
        self.processor = processor
        self.decoder_start_token_id = decoder_start_token_id

    def __call__(self, features) -> Dict[str, torch.Tensor]:
        """
        Pads and batches input features and labels for speech-to-text model training.

        Args:
            features: List of feature dictionaries, each containing "input_features" and "labels".

        Returns:
            Dict[str, torch.Tensor]: Dictionary with batched and padded "input_features" and "labels".
        """
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch          = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels         = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Remove bos token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollator(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id,
)

### 6 - Defining the Metrics

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Computes the Word Error Rate (WER) metric for ASR model predictions.

    Args:
        pred: An object with `predictions` (model output IDs) and `label_ids` (ground truth IDs).

    Returns:
        dict: A dictionary containing the WER score as a percentage under the key "wer".
    """
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # replace -100 back to pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer * 100.0}

### 7 - Loading the Whisper Model

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.language           = "french"
model.generation_config.task               = "transcribe"
model.generation_config.forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(language="french", task="transcribe")

### 8 - Configuring Training Arguments and Initializing the Seq2Seq Trainer

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-kabyle",
    per_device_train_batch_size=36,
    per_device_eval_batch_size=36,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    warmup_steps=200,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    push_to_hub=False,
    report_to="none",
    fp16=True,
    # gradient_checkpointing=True # Optional: enable if memory is a problem
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics,
)

### 9 - Training

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
trainer.train()

**Output:**

`[1824/1824 3:31:51, Epoch 2/2]`

| Epoch | Training Loss | Validation Loss | Wer |
| :---: | :---: | :---: | :---: |
| 1 | 0.291500 | 0.280929 | 39.424823 |
| 2 | 0.138700 | 0.222439 | 35.422214 |
    
```python
TrainOutput(
    global_step=1824, 
    training_loss=0.5365522595351202, 
    metrics={
        'train_runtime': 12720.2406, 
        'train_samples_per_second': 5.159, 
        'train_steps_per_second': 0.143, 
        'total_flos': 1.893870548140032e+19, 
        'train_loss': 0.5365522595351202, 
        'epoch': 2.0
    }
)

### 10 - Export the Model's Training Files to the Cloud (Google Drive)

In [ ]:
drive.mount('/content/drive')

src = '/content/whisper-kabyle/'
dst = '/content/drive/MyDrive/Models/'

os.makedirs(os.path.dirname(dst), exist_ok=True)

cmd = ['rsync', '-avh', '--progress', src, dst]
subprocess.run(cmd, check=True)

print(f"r-synced {src} -> {dst}")

### 11 - Shutdown Google Colab's runtime

In [ ]:
runtime.unassign()

## Part Two: Inference

### 1 - Loading the Trained Model from Google Drive
Mounting Google Drive with Google Colab

In [ ]:
drive.mount('/content/drive')

checkpoint_path = "/content/drive/MyDrive/Models/checkpoint-1824"
model_name      = "openai/whisper-small"

model     = WhisperForConditionalGeneration.from_pretrained(checkpoint_path)
processor = WhisperProcessor.from_pretrained(model_name)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

### 2 - Defining the Pipline and the Transcription Function

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1,
)

def transcribe_audio_file(path):
    """
    Transcribes an audio file to text using the ASR pipeline.

    Args:
        path (str): Path to the audio file to be transcribed.

    Returns:
        str: The transcribed text from the audio file.
    """
    result = pipe(path)
    return result["text"]

### 3 - Executing Inference on a WAV File

In [ ]:
wav_filepath = "/content/A001_D01_4.wav"
print(transcribe_audio_file(wav_filepath))

Device set to use cuda:0

Anta iṛuḥ baba sredi-d-tuh!

#### Actual Transcription:
Anda iruḥ baba-s ad yeddu